# 1) Eddy ID and tracking

The aim of this notebook is to run parallelised eddy ID and tracking on model data on levante (dkrz). 

This notebook uses the "updated" version of py-eedy-tracker that was improved by Aaron Wienkers during the first eerie hackathon (Nov 2023). Link to the py-eddy-tracker updated version and how to use it - https://github.com/eerie-project/EERIE_hackathon_2023/tree/main/RESULTS/pyeddytracker_xarray_dask_parallel. The updated version is able to use intake catalongs and xarrays.


Most of the Eddy ID and tracking functions come from py-eddy-tracker (https://py-eddy-tracker.readthedocs.io/en/stable/) and a lot of other eddy ID examples from EERIE first hackathon (Nov 2023) (https://github.com/eerie-project/EERIE_hackathon_2023/tree/main/RESULTS). 


**Input: Regridded 0.25deg ICON historical run (because it is regridded, any model should work), 1993-2014**

**Output: .nc files of the standard py-eddy-tracker cyclonic and anticyclonis _tracked, _short and _untracked eddy trajecries**


Date created: 12.12.2023

Last edited: 31.10.2025

Stella Bērziņa (stella.berzina@usys.ethz.ch)

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pylab as plt
import matplotlib.cm as cm
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator
import glob
import intake
from pathlib import Path
import dask
import pandas as pd
dask.config.set({"array.slicing.split_large_chunks": True}) 
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
import sys
from numpy import arange, ones

import matplotlib as mpl
import matplotlib.lines as mlines
from matplotlib.cm import get_cmap
from matplotlib import colors 
from matplotlib.colors import ListedColormap
from matplotlib.figure import Figure

from py_eddy_tracker.observations.network import NetworkObservations
from py_eddy_tracker.observations.observation import EddiesObservations, Table
from py_eddy_tracker.observations.tracking import TrackEddiesObservations
from py_eddy_tracker.dataset.grid import RegularGridDataset
from py_eddy_tracker.featured_tracking.area_tracker import AreaTracker
from py_eddy_tracker.gui import GUI
from py_eddy_tracker.tracking import Correspondances


from datetime import datetime, timedelta
from netCDF4 import Dataset

import io
import os

import warnings
warnings.filterwarnings("ignore")

# from concurrent.futures import ProcessPoolExecutor
# from concurrent.futures import ThreadPoolExecutor

import dask
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

import cartopy.crs as ccrs
from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter,LatitudeLocator)
import cartopy.feature as cfeature


# Data

In [2]:
# icon data i use for a historical run
eerie_cat=intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
da = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["gr025"]["2d_daily_mean"].to_dask()

In [3]:
da

<xarray.Dataset>
Dimensions:                              (time: 23741, lat: 721, lon: 1440,
                                          lev: 1, depth: 1)
Coordinates:
  * depth                                (depth) float64 1.0
  * lat                                  (lat) float64 -90.0 -89.75 ... 90.0
  * lev                                  (lev) float64 0.0
  * lon                                  (lon) float64 0.0 0.25 ... 359.5 359.8
  * time                                 (time) datetime64[ns] 1950-01-01T23:...
Data variables: (12/34)
    FrshFlux_IceSalt                     (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    FrshFlux_TotalIce                    (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    Wind_Speed_10m                       (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Evaporation    (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Precipitation  (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Runoff         (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    ...                                   ...
    so                                   (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    ssh                                  (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    stretch_c                            (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    to                                   (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    u                                    (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    v                                    (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
Attributes: (12/31)
    Conventions:           CF-1.7 CMIP-6.2
    activity_id:           HighResMIP
    data_specs_version:    01.00.32
    forcing_index:         1
    initialization_index:  1
    license:               EERIE model data produced by MPI-M is licensed und...
    ...                    ...
    parent_activity_id:    HighResMIP
    sub_experiment_id:     none
    experiment:            coupled historical 1950-2014
    source:                ICON-ESM-ER (2023): \naerosol: none, prescribed MA...
    institution:           Max Planck Institute for Meteorology, Hamburg 2014...
    sub_experiment:        none

In [7]:
"""
Relevant if you want to run analysis on a subset of data. I am using all the time steps here.
"""
# 2022-2032
# ds_subset = da.sel(time=slice('1993-01-02','2014-12-31'))
ds_subset = da.sel(time=slice('1993-01-01','2014-12-31'))

datearr = [pd.Timestamp(t).to_pydatetime() for t in ds_subset.time.values]
ds_subset

<xarray.Dataset>
Dimensions:                              (time: 8035, lat: 721, lon: 1440,
                                          lev: 1, depth: 1)
Coordinates:
  * depth                                (depth) float64 1.0
  * lat                                  (lat) float64 -90.0 -89.75 ... 90.0
  * lev                                  (lev) float64 0.0
  * lon                                  (lon) float64 0.0 0.25 ... 359.5 359.8
  * time                                 (time) datetime64[ns] 1993-01-01T23:...
Data variables: (12/34)
    FrshFlux_IceSalt                     (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    FrshFlux_TotalIce                    (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    Wind_Speed_10m                       (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Evaporation    (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Precipitation  (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Runoff         (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    ...                                   ...
    so                                   (time, depth, lat, lon) float32 dask.array<chunksize=(6, 1, 721, 1440), meta=np.ndarray>
    ssh                                  (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    stretch_c                            (time, lat, lon) float32 dask.array<chunksize=(6, 721, 1440), meta=np.ndarray>
    to                                   (time, depth, lat, lon) float32 dask.array<chunksize=(6, 1, 721, 1440), meta=np.ndarray>
    u                                    (time, depth, lat, lon) float32 dask.array<chunksize=(6, 1, 721, 1440), meta=np.ndarray>
    v                                    (time, depth, lat, lon) float32 dask.array<chunksize=(6, 1, 721, 1440), meta=np.ndarray>
Attributes: (12/31)
    Conventions:           CF-1.7 CMIP-6.2
    activity_id:           HighResMIP
    data_specs_version:    01.00.32
    forcing_index:         1
    initialization_index:  1
    license:               EERIE model data produced by MPI-M is licensed und...
    ...                    ...
    parent_activity_id:    HighResMIP
    sub_experiment_id:     none
    experiment:            coupled historical 1950-2014
    source:                ICON-ESM-ER (2023): \naerosol: none, prescribed MA...
    institution:           Max Planck Institute for Meteorology, Hamburg 2014...
    sub_experiment:        none

In [5]:
ds=ds_subset

# Eddy ID

In [8]:
# Directories
scratch = '/scratch/b/b382618/historical/icon/'
# expid = 'eerie-control-1950-v20240618'
gridspec = 'gr025'
varname = 'ssh'
# varname = 'avg_zos'

datadir = scratch
print(datadir)

eddy_dir = datadir+'/eddyID/'
if not os.path.exists(eddy_dir):
    os.makedirs(eddy_dir)

track_dir = datadir+'/eddyTracks/'
if not os.path.exists(track_dir):
    os.makedirs(track_dir)

/scratch/b/b382618/icon/


### Start SLURMCluster on Levante for Parallel Computing

In [7]:
# define cluster; corresponds to one SLURM job
cluster = SLURMCluster(name='dask-cluster',
                      cores=16,  
                      memory='256GB',
                      processes=16,
                      interface='ib0',
                      queue='compute',
                      account='bk1377',
                      walltime='06:00:00',
                      asynchronous=0,
                      log_directory='/home/b/b382618/.log_trash')

In [8]:
# multiply cluster size by 2; equivalent to 2 SLURM jobs
# this will actually open 2 jobs on squeue

cluster.scale(cores=32)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.128.8.79:8787/status,
Dashboard: http://10.128.8.79:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.128.8.79:38231,Workers: 0
Dashboard: http://10.128.8.79:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


##### Eddy ID parameters used by AVISO

In [9]:
wavelength = 700 #choice of spatial cutoff for high pass filter in km
step_ht = 0.002 #intervals to search for closed contours (2mm in this case)
pixel_limit = None  # Min and max pixel count for valid contour
shape_error = 70  # Error max (%) between ratio of circle fit and contour

In [14]:
def detection(ncfile, varname, date): 
    # g = RegularGridDataset(None, "lon", "lat", centered=True, nc4file=ncfile)  # NOTE: Using 'None' for the .nc file path then requires specifying directly the netcdf4 variable in memory
    g = RegularGridDataset(None, "lon", "lat", centered=True, nc4file=ncfile)  # NOTE: Using 'None' for the .nc file path then requires specifying directly the netcdf4 variable in memory

    g.add_uv(varname)
    g.bessel_high_filter(varname, wavelength, order=1)
        
    a, c = g.eddy_identification(varname, "u", "v", 
    date,  # Date of identification
    step_ht,  # step between two isolines of detection (m)
    pixel_limit=pixel_limit,  # Min and max pixel count for valid contour
    shape_error=70  # Error max (%) between ratio of circle fit and contour
    )
    print("id done")
    return a,c,g


In [15]:
def detection_save_netcdf_output(date, tt):
    # Load data from xarray into netcdf4 type
    da_ssh = ds.isel(time=tt).ssh.drop('time')
    da_ssh.load(scheduler='sync')
    da_netcdf = Dataset('in-mem-file', mode='r', memory=da_ssh.to_netcdf())
    
    print('Identifying daily eddies for '+date.strftime('%Y%m%d'))
    a_filtered, c_filtered, g_filtered = detection(da_netcdf,varname,date)
    with Dataset(date.strftime(eddy_dir+"eddyID_anticyclonic_"+date.strftime('%Y%m%d')+".nc"), "w") as h:
        a_filtered.to_netcdf(h)
    with Dataset(date.strftime(eddy_dir+"eddyID_cyclonic_"+date.strftime('%Y%m%d')+".nc"), "w") as h:
        c_filtered.to_netcdf(h)
    del a_filtered
    del c_filtered
    del g_filtered
    del date

In [ ]:
lazy_results = []
for i in range(len(datearr)): 
    lazy_result = dask.delayed(detection_save_netcdf_output)(date=datearr[i], tt=int(i))
    lazy_results.append(lazy_result)  
    
futures = dask.compute(*lazy_results)
results = dask.compute(*futures)

In [3]:
client.close()
client.shutdown()

NameError: name 'client' is not defined

# Eddy tracking

In [3]:
from py_eddy_tracker.featured_tracking.area_tracker import AreaTracker
from py_eddy_tracker.tracking import Correspondances

##### Eddy tracking parameters used by AVISO

In [4]:
nb_obs_min = 4 # minimum of 10 points in track to be considered a long trajectory
raw = False # 
cmin = 0.05 # minimum contour
virtual = 4 # number of consecutive timesteps with missing detection allowed
class_kw = dict(cmin=cmin)
zarr = False

In [5]:
# if the number of cores is increased then not enough memory per worker to deal with 37 years of data

cluster = SLURMCluster(name='dask-cluster',
                      cores=4,
                      memory='512GB',
                      processes=4,
                      interface='ib0',
                      queue='compute',
                      account='bk1377',
                      walltime='05:00:00',
                      asynchronous=0,
                      log_directory='/home/b/b382618/.log_trash')

In [6]:
cluster.scale(cores=4)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.128.8.74:8787/status,
Dashboard: http://10.128.8.74:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.128.8.74:34023,Workers: 0
Dashboard: http://10.128.8.74:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [7]:
#Functions from eddy-tracking.py (aided by Malcolm Roberts)

def tracking(file_objects, previous_correspondance, eddy_type, zarr=False, nb_obs_min=10, raw=True, cmin=0.05, virtual=4):
    # We run a tracking with a tracker which uses contour overlap, on first time step
    output_dir = os.path.dirname(previous_correspondance)
    class_kw = dict(cmin=cmin)
    if not os.path.isfile(previous_correspondance):
        c = Correspondances(
            datasets=file_objects, class_method=AreaTracker, 
            class_kw=class_kw, virtual=virtual
        )
        c.track()
        c.prepare_merging()
    else:
        c = Correspondances(
            datasets=file_objects, class_method=AreaTracker, 
            class_kw=class_kw, virtual=virtual,
            previous_correspondance=previous_correspondance
        )
        c.track()
        c.prepare_merging()
        c.merge()

    new_correspondance = previous_correspondance[:-3]+'_new.nc'
    with Dataset(new_correspondance, "w") as h:
        c.to_netcdf(h)

    try:
        # test can read new file, and then move to replace old file
        nc = Dataset(new_correspondance, 'r')
        os.rename(new_correspondance, previous_correspondance)
    except:
        raise Exception('Error opening new correspondance file '+new_correspondance)

    write_obs_files(c, raw, output_dir, zarr, eddy_type, nb_obs_min)
    

def write_obs_files(c, raw, output_dir, zarr, eddy_type, nb_obs_min):
    kw_write = dict(path=output_dir, zarr_flag=zarr, sign_type=eddy_type)

    fout = os.path.join(output_dir, eddy_type+'_untracked.nc')
    c.get_unused_data(raw_data=raw).write_file(
        filename=fout
    )

    short_c = c._copy()
    short_c.shorter_than(size_max=nb_obs_min)
    short_track = short_c.merge(raw_data=raw)

    if c.longer_than(size_min=nb_obs_min) is False:
        long_track = short_track.empty_dataset()
    else:
        long_track = c.merge(raw_data=raw)

    # We flag obs
    if c.virtual:
        long_track["virtual"][:] = long_track["time"] == 0
        long_track.normalize_longitude()
        long_track.filled_by_interpolation(long_track["virtual"] == 1)
        short_track["virtual"][:] = short_track["time"] == 0
        short_track.normalize_longitude()
        short_track.filled_by_interpolation(short_track["virtual"] == 1)

    print("Longer track saved have %d obs", c.nb_obs_by_tracks.max())
    print(
        "The mean length is %d observations for long track",
        c.nb_obs_by_tracks.mean(),
    )

    fout = os.path.join(output_dir, eddy_type+'_tracks.nc')
    long_track.write_file(filename=fout)
    fout = os.path.join(output_dir, eddy_type+'_short.nc')
    short_track.write_file(
        #filename="%(path)s/%(sign_type)s_track_too_short.nc", **kw_write
        filename=fout
    )



def tracking_eddytype(eddy_type, track_dir, eddy_dir, zarr, nb_obs_min, raw, cmin):
    previous_correspondance = os.path.join(track_dir, eddy_type+'_correspondance.nc')
    search = os.path.join(eddy_dir+'eddyID_'+eddy_type+'_'+'????????.nc')
    print('search files ',search)
    file_objects = sorted(glob.glob(search))
    tracking(file_objects, previous_correspondance, eddy_type, zarr=zarr, nb_obs_min=nb_obs_min, raw=raw, cmin=cmin)


In [8]:
lazy_results = []
print('eddy_dir = ', eddy_dir)
print('track_dir = ', track_dir)
for eddy_type in ['cyclonic','anticyclonic']:
    lazy_result = dask.delayed(tracking_eddytype)(track_dir=track_dir,
                                                    eddy_dir=eddy_dir,
                                                    eddy_type=eddy_type,
                                                    zarr=zarr, nb_obs_min=nb_obs_min,
                                                    raw=raw, cmin=cmin)
    lazy_results.append(lazy_result)  

futures = dask.compute(*lazy_results)
results = dask.compute(*futures)

eddy_dir =  /scratch/b/b382618/avisoadt//eddyID/
track_dir =  /scratch/b/b382618/avisoadt//eddyTracks/


In [14]:
client.close()
client.shutdown()